# Data Enrichment

1. Read the concatinated data
2. Handle Missing values
3. Create derived columns for further analyssi
4. Save the data in `data/JC`


[Step 7](https://hovhannisyan91.github.io/aca/materials/python/session15.html#step-7-setting-dates)-[Step 11](https://hovhannisyan91.github.io/aca/materials/python/session15.html#step-11-time-based-variables)

## Loading Libraries

In [12]:

import requests

import numpy as np
import pandas as pd

import plotly.express as px
import folium

import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

In [13]:
OUTPUT_DIR = "../data/citibike/"

In [14]:
citibike_df = pd.read_csv("../data/citibike/JC/JC2022.csv")
citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,B7BCBC133222B04A,classic_bike,2022-02-17 11:48:16,2022-02-17 11:58:30,Astor Place,JC077,Journal Square,JC103,40.719282,-74.071262,40.733670,-74.062500,member
1,0F9F1A0F18FD3A22,electric_bike,2022-02-02 08:31:26,2022-02-02 08:38:08,Astor Place,JC077,Sip Ave,JC056,40.719282,-74.071262,40.730897,-74.063913,member
2,44B5D106DDB509AC,electric_bike,2022-02-09 14:05:49,2022-02-09 14:10:21,Astor Place,JC077,Sip Ave,JC056,40.719282,-74.071262,40.730897,-74.063913,member
3,E1434F258C195DC6,classic_bike,2022-02-08 07:57:44,2022-02-08 08:15:26,Astor Place,JC077,Sip Ave,JC056,40.719282,-74.071262,40.730897,-74.063913,member
4,64821879B2659E35,classic_bike,2022-02-07 08:05:28,2022-02-07 08:24:03,Astor Place,JC077,Sip Ave,JC056,40.719282,-74.071262,40.730897,-74.063913,member


## Observe Missing Values

In [15]:
# PUT your CODE
missing_values = (
    citibike_df
    .isna()
    .sum()
    .reset_index()
)

missing_values.columns = ["column", "missing_count"]

missing_values["missing_share"] = (
    missing_values["missing_count"] / len(citibike_df)
)

missing_values.sort_values("missing_count", ascending=False)

,column,missing_count,missing_share
6,end_station_name,2414,0.003175
7,end_station_id,2414,0.003175
11,end_lng,1471,0.001935
10,end_lat,1471,0.001935
4,start_station_name,10,0.000013
5,start_station_id,10,0.000013
2,started_at,0,0.000000
1,rideable_type,0,0.000000
0,ride_id,0,0.000000
3,ended_at,0,0.000000


## Derived Columns

1. `ride_duration_min`
2. filter out anomalies
3. drop missing values
4. `date`
5. `month` 
6. `month_name`
7. `month_number` 
8. `day_of_week`
9. `hour` 
10. `season`

> all the calculations must be based on `started_at` column
>
> REMEMBER: first to convert it into datetime type

In [16]:
citibike_df["started_at"] = pd.to_datetime(citibike_df["started_at"])
citibike_df["ended_at"] = pd.to_datetime(citibike_df["ended_at"])

citibike_df["ride_duration_minutes"] = (
    citibike_df["ended_at"] - citibike_df["started_at"]
).dt.total_seconds() / 60

In [17]:
citibike_df = citibike_df.dropna(
    subset=[
        "ride_id",
        "started_at",
        "ended_at",
        "start_lat",
        "start_lng",
        "end_lat",
        "end_lng"
    ]
)


citibike_df = citibike_df[
    (citibike_df["ride_duration_minutes"] > 1) &
    (citibike_df["ride_duration_minutes"] <= 24 * 60)
].copy()

In [18]:
citibike_df["date"] = citibike_df["started_at"].dt.date
citibike_df["month"] = citibike_df["started_at"].dt.to_period("M").astype(str)
citibike_df["month_name"] = citibike_df["started_at"].dt.month_name()
citibike_df["day_of_week"] = citibike_df["started_at"].dt.day_name()
citibike_df["hour"] = citibike_df["started_at"].dt.hour

In [19]:

def assign_season(month_number):
    if month_number in [12, 1, 2]:
        return "Winter"
    elif month_number in [3, 4, 5]:
        return "Spring"
    elif month_number in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"
    
citibike_df["season"] = (
    citibike_df["started_at"]
    .dt.month
    .apply(assign_season)
)

In [20]:
citibike_df[
    [
        "started_at",
        "date",
        "month",
        "month_name",
        "day_of_week",
        "hour",
        "season"
    ]
].head()

,started_at,date,month,month_name,day_of_week,hour,season
0,2022-02-17 11:48:16,2022-02-17,2022-02,February,Thursday,11,Winter
1,2022-02-02 08:31:26,2022-02-02,2022-02,February,Wednesday,8,Winter
2,2022-02-09 14:05:49,2022-02-09,2022-02,February,Wednesday,14,Winter
3,2022-02-08 07:57:44,2022-02-08,2022-02,February,Tuesday,7,Winter
4,2022-02-07 08:05:28,2022-02-07,2022-02,February,Monday,8,Winter


In [ ]:
citibike_df.to_csv("../data/citibike/JC/JC2023_Enriched.csv", index=False)

## Store the data 

In [22]:
# PUT YOUR CODE